# Modul 09: scikit-learn-Estimator-API und Pipelines | Lösungen

## Überblick

Sie untersuchen Datensatzobjekte und die einheitliche Estimator-Schnittstelle von scikit-learn. Anschließend bauen Sie eine leakage-freie Pipeline für gemischte numerische und kategoriale Daten mit Imputation, Skalierung, One-Hot-Kodierung und einem linearen Modell.

**Zugehörige Vorlesungen**

- **Estimator-API**
- **Pipelines bauen**

## Lernziele

Nach der Bearbeitung können Sie:

- fit, transform, predict, Hyperparameter und gelernte Attribute voneinander unterscheiden.
- Dummy- und lineare Modelle auf sauberen Splits trainieren und fair vergleichen.
- ColumnTransformer und Pipeline für fehlende numerische und kategoriale Werte korrekt verbinden und inspizieren.

## Geprüfte Fähigkeiten

- Bunch-Datensätze, Estimator-Parameter und gelernte Attribute
- SimpleImputer, StandardScaler, OneHotEncoder und ColumnTransformer
- Pipeline, named_steps, verschachtelte Parameter und Merkmalsnamen

## Hinweise zur Bearbeitung

Dieses Lösungsnotebook enthält dieselben Aufgaben wie das Übungsnotebook sowie vollständige, ausführlich kommentierte Musterlösungen. Bearbeiten Sie nach Möglichkeit zuerst das Übungsnotebook und nutzen Sie dieses Dokument anschließend zur Kontrolle und Vertiefung.

- **Erwarteter Schwierigkeitsgrad:** mittel
- Verwenden Sie sprechende Variablennamen und prüfen Sie wichtige Zwischenformen und Wertebereiche.
- Verändern Sie die vorgegebenen Zufalls-Startwerte nur, wenn eine Aufgabe dies ausdrücklich verlangt.
- Interpretieren Sie Ergebnisse fachlich. Eine einzelne Kennzahl ist selten eine vollständige Begründung.
- Alle Aufgaben sind für die kostenlose Google-Colab-Umgebung ausgelegt. Die Datensätze und Modelle sind bewusst klein gehalten. Eine GPU ist nicht erforderlich, kann aber bei einzelnen Deep-Learning-Aufgaben die Laufzeit verkürzen.

## Einrichtung und gemeinsame Datenbasis

Neben dem Iris-Bunch wird ein kleiner gemischter Wartungsdatensatz mit Fehlwerten und seltenen Kategorien erzeugt. Eine unbekannte Kategorie im Test prüft die Robustheit des Encoders.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.datasets import load_iris
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_SEED = 42

iris = load_iris()

wartung = pd.DataFrame(
    {
        "temperatur": [62, 71, 68, np.nan, 85, 79, 66, 91, 74, 69, 88, 64, 77, 82, 73, 70, 95, 67],
        "vibration": [1.2, 1.8, np.nan, 1.5, 3.1, 2.6, 1.3, 3.7, 2.0, 1.6, 3.3, 1.1, 2.2, 2.9, 1.9, 1.4, 4.0, 1.5],
        "stunden": [120, 250, 190, 210, 480, 390, 150, 520, 300, 230, 450, 100, 340, 410, 280, 170, 560, 200],
        "standort": ["Nord", "Süd", "Nord", "West", "Süd", "Nord", "West", "Süd", "Nord", "West", "Süd", "Nord", "West", "Nord", "Süd", "West", "Ost", None],
        "typ": ["A", "A", "B", "A", "C", "B", "A", "C", "B", "A", "C", "A", "B", "C", "B", "A", "C", "B"],
        "ausfall": [0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0],
    }
)

print("Einrichtung abgeschlossen.")
print("scikit-learn-Version:", sklearn.__version__)
print("Gemischte Tabelle:", wartung.shape)

### Aufgabe 1: Bunch-Datensatz und Datenrollen untersuchen

1. Geben Sie die verfügbaren Schlüssel des Iris-Bunch aus.
2. Prüfen Sie Formen von `data` und `target`, Merkmalsnamen und Klassennamen.
3. Erzeugen Sie einen beschrifteten DataFrame mit Zielspalte `art`.
4. Erklären Sie, welche Informationen Metadaten und welche eigentliche Lernwerte sind.

In [ ]:
# iris wurde in der Setup-Zelle geladen.

# ============================================================
# MUSTERLÖSUNG
# ============================================================

print("Bunch-Schlüssel:", iris.keys())
print("Datenform:", iris.data.shape)
print("Zielform:", iris.target.shape)
print("Merkmalsnamen:", iris.feature_names)
print("Klassennamen:", iris.target_names)

iris_df = pd.DataFrame(iris.data, columns=iris.feature_names)
iris_df["art"] = pd.Categorical.from_codes(iris.target, iris.target_names)
display(iris_df.head())

# Grundprüfung: Eine Zielangabe gehört zu jeder Beobachtungszeile.
assert len(iris.data) == len(iris.target)

> **Musterantwort und Interpretation**
>
> Die numerische Eingabematrix `data` und der Zielvektor `target` werden an `fit` übergeben. Merkmalsnamen, Zielnamen und DESCR dienen dem Datenverständnis, der Dokumentation und späteren Interpretation, sind aber nicht automatisch Lernwerte.

### Aufgabe 2: fit, transform, predict, Parameter und Attribute unterscheiden

Verwenden Sie einen `StandardScaler` und eine `LogisticRegression` auf Iris:

1. Teilen Sie die Daten stratifiziert.
2. Zeigen Sie ausgewählte Parameter mit `get_params()` vor dem Training.
3. Passen Sie den Skalierer nur auf Training an und transformieren Sie beide Splits.
4. Trainieren Sie das Modell und geben Sie gelernte Attribute wie `mean_`, `scale_`, `coef_` und `classes_` aus.
5. Berechnen Sie Testgenauigkeit.

In [ ]:
X_iris_train, X_iris_test, y_iris_train, y_iris_test = train_test_split(
    iris.data,
    iris.target,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=iris.target,
)

# ============================================================
# MUSTERLÖSUNG
# ============================================================

skalierer = StandardScaler()
modell = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)

print("Scaler-Parameter vor fit:", {k: v for k, v in skalierer.get_params().items()})
print("Ausgewählte Modellparameter:", {k: modell.get_params()[k] for k in ["C", "max_iter", "solver"]})

X_iris_train_s = skalierer.fit_transform(X_iris_train)
X_iris_test_s = skalierer.transform(X_iris_test)
modell.fit(X_iris_train_s, y_iris_train)

print("Gelernte Mittelwerte:", np.round(skalierer.mean_, 3))
print("Gelernte Skalen:", np.round(skalierer.scale_, 3))
print("Koeffizientenform:", modell.coef_.shape)
print("Gelernte Klassen:", modell.classes_)

iris_pred = modell.predict(X_iris_test_s)
print(f"Testgenauigkeit: {accuracy_score(y_iris_test, iris_pred):.3f}")

> **Musterantwort und Interpretation**
>
> Der Unterstrich kennzeichnet Konventionen zufolge Werte, die erst durch `fit` aus Daten gelernt wurden. Hyperparameter ohne Unterstrich werden dagegen vor dem Training festgelegt. Der Unterschied hilft, Modellkonfiguration und gelernte Zustände auseinanderzuhalten.

### Aufgabe 3: Dummy-Baseline und lineares Modell vergleichen

1. Trainieren Sie auf dem Iris-Split einen `DummyClassifier(strategy="most_frequent")`.
2. Vergleichen Sie dessen Accuracy mit der logistischen Regression.
3. Erstellen Sie eine kleine Ergebnisgrafik.
4. Erklären Sie, welchen zusätzlichen Nutzen das echte Modell gegenüber der Baseline zeigt.

In [ ]:
# Nutzen Sie X_iris_train_s, X_iris_test_s und die Zielwerte aus Aufgabe 2.

# ============================================================
# MUSTERLÖSUNG
# ============================================================

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_iris_train_s, y_iris_train)
dummy_pred = dummy.predict(X_iris_test_s)

ergebnisse_iris = pd.DataFrame(
    {
        "Modell": ["Mehrheitsbaseline", "Logistische Regression"],
        "Accuracy": [accuracy_score(y_iris_test, dummy_pred), accuracy_score(y_iris_test, iris_pred)],
    }
)
display(ergebnisse_iris.round(3))

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(ergebnisse_iris["Modell"], ergebnisse_iris["Accuracy"])
ax.set_ylim(0, 1)
ax.set_title("Baseline und gelerntes Modell")
ax.set_ylabel("Accuracy")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

> **Musterantwort und Interpretation**
>
> Sie zeigt, welche Leistung ohne Nutzung der Merkmale erreichbar ist. Ein komplexeres Modell muss diesen Mindestvergleich klar übertreffen, sonst liefert es keinen nachgewiesenen Zusatznutzen. Baselines helfen außerdem, Fehler in Split, Zielvariable oder Metrik früh zu erkennen.

### Aufgabe 4: Spaltenspezifische Vorverarbeitung bauen

Teilen Sie `wartung` stratifiziert in Training und Test. Bauen Sie anschließend:

- numerische Pipeline: Median-Imputation und Standardisierung,
- kategoriale Pipeline: häufigste Kategorie und One-Hot-Encoding,
- `OneHotEncoder(handle_unknown="ignore", min_frequency=2)`,
- `ColumnTransformer`, der beide Spaltentypen zusammenführt.

Passen Sie den Transformer nur auf Training an und prüfen Sie die resultierenden Formen.

In [ ]:
numerische_spalten = ["temperatur", "vibration", "stunden"]
kategoriale_spalten = ["standort", "typ"]

X_wartung = wartung[numerische_spalten + kategoriale_spalten]
y_wartung = wartung["ausfall"]

X_w_train, X_w_test, y_w_train, y_w_test = train_test_split(
    X_wartung,
    y_wartung,
    test_size=0.28,
    random_state=RANDOM_SEED,
    stratify=y_wartung,
)

# ============================================================
# MUSTERLÖSUNG
# ============================================================

# Die Reihenfolge innerhalb jeder Teilpipeline ist fachlich wichtig.
numerische_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

kategoriale_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=2,
                sparse_output=False,
            ),
        ),
    ]
)

vorverarbeitung = ColumnTransformer(
    transformers=[
        ("num", numerische_pipeline, numerische_spalten),
        ("cat", kategoriale_pipeline, kategoriale_spalten),
    ]
)

X_w_train_t = vorverarbeitung.fit_transform(X_w_train)
X_w_test_t = vorverarbeitung.transform(X_w_test)

print("Rohform Training:", X_w_train.shape)
print("Transformierte Form Training:", X_w_train_t.shape)
print("Transformierte Form Test:", X_w_test_t.shape)
assert X_w_train_t.shape[1] == X_w_test_t.shape[1]

> **Musterantwort und Interpretation**
>
> Scaler und viele Modelle können fehlende Werte nicht sinnvoll verarbeiten, und Kategorien müssen vor dem Encoder als definierte Werte vorliegen. Die Imputation erzeugt zunächst eine vollständige Spalte; danach kann die passende numerische oder kategoriale Transformation konsistent angewendet werden.

### Aufgabe 5: End-to-End-Pipeline trainieren und robuste Vorhersage prüfen

1. Verbinden Sie `vorverarbeitung` und `LogisticRegression` in einer Pipeline.
2. Trainieren Sie direkt mit dem rohen Trainings-DataFrame.
3. Berechnen Sie Testgenauigkeit und Wahrscheinlichkeiten.
4. Erstellen Sie einen neuen Fall mit unbekanntem Standort `Zentral` und fehlender Vibration und führen Sie eine Vorhersage aus.
5. Erklären Sie, warum `handle_unknown="ignore"` wichtig ist.

In [ ]:
# Nutzen Sie die Vorverarbeitung aus Aufgabe 4.

# ============================================================
# MUSTERLÖSUNG
# ============================================================

gesamt_pipeline = Pipeline(
    steps=[
        ("vorverarbeitung", vorverarbeitung),
        ("modell", LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)),
    ]
)

gesamt_pipeline.fit(X_w_train, y_w_train)
wartung_pred = gesamt_pipeline.predict(X_w_test)
wartung_prob = gesamt_pipeline.predict_proba(X_w_test)[:, 1]

print(f"Testgenauigkeit: {accuracy_score(y_w_test, wartung_pred):.3f}")
print("Testwahrscheinlichkeiten:", np.round(wartung_prob, 3))

neuer_fall = pd.DataFrame(
    {
        "temperatur": [86.0],
        "vibration": [np.nan],
        "stunden": [430],
        "standort": ["Zentral"],
        "typ": ["C"],
    }
)

neue_klasse = gesamt_pipeline.predict(neuer_fall)[0]
neue_wahrscheinlichkeit = gesamt_pipeline.predict_proba(neuer_fall)[0, 1]
print("Neue Klasse:", neue_klasse)
print("Ausfallwahrscheinlichkeit:", round(neue_wahrscheinlichkeit, 3))

> **Musterantwort und Interpretation**
>
> Mit `handle_unknown='ignore'` erhält sie in den während des Trainings bekannten Standortspalten ausschließlich Nullen. Die Pipeline bricht dadurch nicht ab. Die Vorhersage kann dennoch unsicher oder systematisch verzerrt sein, weshalb unbekannte Kategorien überwacht und fachlich bewertet werden müssen.

### Aufgabe 6: Pipeline-Schritte, Parameter und Merkmalsnamen inspizieren

1. Geben Sie `named_steps` der Gesamtpipeline aus.
2. Lesen Sie den gelernten numerischen Median und die gelernten Kategorien aus verschachtelten Schritten aus.
3. Rufen Sie `get_feature_names_out()` auf.
4. Erstellen Sie eine Tabelle aus transformiertem Merkmalsnamen und Modellkoeffizient.
5. Zeigen Sie die fünf betragsmäßig stärksten Koeffizienten.
6. Ändern Sie mit `set_params` den Modellparameter `C` auf 0,5 und trainieren Sie erneut.

In [ ]:
# gesamt_pipeline ist bereits trainiert.

# ============================================================
# MUSTERLÖSUNG
# ============================================================

print("Pipeline-Schritte:", gesamt_pipeline.named_steps.keys())

angepasste_vorverarbeitung = gesamt_pipeline.named_steps["vorverarbeitung"]
num_imputer = angepasste_vorverarbeitung.named_transformers_["num"].named_steps["imputer"]
cat_encoder = angepasste_vorverarbeitung.named_transformers_["cat"].named_steps["encoder"]

print("Gelernte numerische Medianwerte:", num_imputer.statistics_)
print("Gelernte Kategorien je Spalte:", cat_encoder.categories_)

merkmalsnamen = angepasste_vorverarbeitung.get_feature_names_out()
koeffizienten = gesamt_pipeline.named_steps["modell"].coef_.ravel()
koeffizienten_df = pd.DataFrame(
    {
        "Merkmal": merkmalsnamen,
        "Koeffizient": koeffizienten,
        "Absoluter_Koeffizient": np.abs(koeffizienten),
    }
).sort_values("Absoluter_Koeffizient", ascending=False)
display(koeffizienten_df.head(5))

# Verschachtelte Parameter werden mit Schrittname__Parametername adressiert.
gesamt_pipeline.set_params(modell__C=0.5)
gesamt_pipeline.fit(X_w_train, y_w_train)
print("Neues C:", gesamt_pipeline.named_steps["modell"].C)
print("Neue Testgenauigkeit:", accuracy_score(y_w_test, gesamt_pipeline.predict(X_w_test)))

> **Musterantwort und Interpretation**
>
> Koeffizienten hängen von Skalierung, Kodierung, Korrelationen zwischen Merkmalen, Regularisierung und Datenauswahl ab. Sie beschreiben eine modellinterne Beziehung unter Konstanthaltung anderer Merkmale, beweisen aber weder Ursache noch stabile Bedeutung außerhalb der betrachteten Daten.

## Abschlusskontrolle

Prüfen Sie vor dem Abschluss:

- Lassen sich alle Zellen in sinnvoller Reihenfolge ausführen?
- Sind Formen, Datentypen, Wertebereiche und Zufalls-Startwerte dokumentiert?
- Wurden Trainings-, Validierungs- und Testinformationen sauber getrennt?
- Sind Diagramme und Kennzahlen beschriftet und fachlich interpretiert?
- Können Sie erklären, warum die gewählten Methoden zur Aufgabenstellung passen?